# Chapter 11 — Support Vector Machines (SVM)

> Tóm tắt kiến thức và thực hành SVM trong *Grokking Machine Learning*.

## 1. Ý tưởng chính

**Support Vector Machine (SVM)** là thuật toán học có giám sát, thường dùng để **phân loại**. SVM tìm **hyperplane** phân chia các lớp và cố gắng **tối đa hóa margin** (khoảng cách giữa hai biên hỗ trợ), đồng thời hạn chế vi phạm margin.

- **Hyperplane / Decision boundary:** Ranh giới quyết định nhãn; trong không gian 2 chiều, linear SVM dùng một đường thẳng.
- **Margin:** Khoảng cách từ hyperplane đến các biên hỗ trợ; với linear SVM chuẩn, tổng độ rộng là `2 / ||w||`.
- **Support vectors:** Các điểm dữ liệu có ảnh hưởng trực tiếp đến nghiệm SVM và vị trí ranh giới.
- **Hard margin:** Không cho phép vi phạm margin; chỉ phù hợp khi dữ liệu có thể tách tuyến tính hoàn toàn.
- **Soft margin:** Cho phép vi phạm margin, nhưng có hình phạt.

## 2. Tham số quan trọng

| Tham số | Ý nghĩa | Điều cần nhớ |
|---|---|---|
| `C` | Hệ số phạt vi phạm margin | **C nhỏ** → regularization mạnh, chấp nhận vi phạm nhiều hơn; **C lớn** → phạt mạnh hơn, có nguy cơ overfit |
| `kernel` | Hàm kernel dùng để xây dựng ranh giới | Thường gặp: `linear`, `poly`, `rbf` |
| `degree` | Bậc của polynomial kernel | Chỉ có tác dụng với `kernel='poly'`; bậc lớn cho phép ranh giới phức tạp hơn, không đảm bảo chính xác hơn |
| `gamma` | Tham số kernel cho `rbf`, `poly`, `sigmoid` | Với RBF: gamma lớn → ảnh hưởng cục bộ hơn, boundary có thể gấp khúc hơn; gamma nhỏ → ảnh hưởng rộng hơn |
| `coef0` | Hằng số trong `poly` và `sigmoid` | Ảnh hưởng mức đóng góp của các bậc đa thức |

**Phân biệt:** `C` không phải bán kính margin; `degree` không phải số lần đường boundary uốn cong.

## 3. Các loại kernel

- **Linear:** `K(x, z) = xᵀz`. Phù hợp khi ranh giới tuyến tính đủ tốt.
- **Polynomial:** `K(x, z) = (γxᵀz + r)^d`, với `d = degree` và `r = coef0`. Có thể tạo boundary phi tuyến; với dữ liệu 2 chiều, bậc kernel xác định bậc tối đa của đa thức biểu diễn hàm quyết định.
- **RBF (Gaussian):** `K(x, z) = exp(-γ||x-z||²)`. Linh hoạt cho nhiều kiểu boundary phi tuyến; `gamma` kiểm soát phạm vi ảnh hưởng.

## 4. Công thức cần hiểu

### Margin của linear SVM

\[
\text{Margin width} = \frac{2}{\lVert w\rVert}
\]

### Hinge loss

\[
L_i = \max(0, 1-y_i f(x_i)), \quad y_i \in \{-1,+1\}
\]

- `y_i f(x_i) ≥ 1`: không có hinge loss.
- `0 < y_i f(x_i) < 1`: đúng phía nhưng nằm trong margin.
- `y_i f(x_i) ≤ 0`: phân loại sai hoặc nằm ngay trên boundary.

### Soft-margin SVM

\[
\min_{w,b,\xi}\;\frac{1}{2}\lVert w\rVert^2 + C\sum_{i=1}^{n}\xi_i
\]

Với ràng buộc `y_i(wᵀx_i + b) ≥ 1 − ξ_i` và `ξ_i ≥ 0`.

**Trực giác:** SVM cân bằng giữa **margin rộng** và **ít vi phạm margin**.

## 5. Code Scikit-learn

### Huấn luyện và đánh giá đúng cách

```python
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

model = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=1.0, gamma='scale')
)
model.fit(X_train, y_train)

print('Train accuracy:', model.score(X_train, y_train))
print('Test accuracy:', model.score(X_test, y_test))
```

> `stratify=labels` giữ tỷ lệ lớp giữa train/test (khi mỗi lớp có đủ mẫu). `StandardScaler` giúp các feature có thang đo tương đồng. Không dùng test set để chọn hyperparameter; hãy dùng validation hoặc cross-validation.

### Thử các giá trị C

```python
for c in [0.01, 0.1, 1, 10, 100]:
    model = SVC(kernel='linear', C=c)
    model.fit(features, labels)
    print(f'C={c}, train accuracy={model.score(features, labels):.4f}')
    plot_model(features, labels, model)  # Nếu đã định nghĩa plot_model
```

### Thử các bậc Polynomial Kernel

```python
for d in [1, 2, 3, 4, 5]:
    model = SVC(kernel='poly', degree=d)
    model.fit(features, labels)
    print(f'degree={d}, train accuracy={model.score(features, labels):.4f}')
    plot_model(features, labels, model)
```

Hai vòng lặp trên chỉ dùng để **quan sát trên tập train**; để so sánh mô hình nghiêm túc, nên chia train/validation/test hoặc dùng `GridSearchCV`.

## 6. Underfitting và Overfitting

| Hiện tượng | Biểu hiện | Gợi ý kiểm tra |
|---|---|---|
| Underfitting | Accuracy train và test đều thấp | Kernel quá đơn giản, `C` quá nhỏ hoặc `gamma` RBF quá nhỏ |
| Overfitting | Accuracy train cao, test thấp | `C` quá lớn, `degree` quá cao hoặc `gamma` RBF quá lớn |
| Generalization tốt | Kết quả trên dữ liệu chưa thấy tốt | Chọn hyperparameter bằng validation/CV |

Các xu hướng trên không tuyệt đối, vì còn phụ thuộc dữ liệu và các tham số khác.

## 7. Checklist ôn tập

- [ ] Giải thích SVM, hyperplane, decision boundary.
- [ ] Giải thích margin và support vectors.
- [ ] Phân biệt hard margin và soft margin.
- [ ] Hiểu `C` và quan hệ nghịch với regularization.
- [ ] Phân biệt linear, polynomial và RBF kernel.
- [ ] Hiểu `degree`, `gamma`, `coef0`.
- [ ] Hiểu ý nghĩa hinge loss và soft-margin objective.
- [ ] Dùng `SVC` với `StandardScaler`, đánh giá train/test và chọn hyperparameter qua validation.

**Nhớ nhanh:** `C` = phạt vi phạm margin; `kernel` = cách xây dựng boundary; `degree` = bậc polynomial kernel; `gamma` = mức ảnh hưởng trong kernel (đặc biệt RBF). **Accuracy trên train không đủ để kết luận mô hình tốt.**
